In [2]:
import os, random, gc, time
from pathlib import Path

# Keep Hugging Face downloads on the local Windows drive, not the network home share.
HF_HOME = Path(os.environ.get('HINGLISH_HATE_HF_HOME',
                              Path(os.environ.get('LOCALAPPDATA', Path.home())) / 'huggingface'))
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Optional Hugging Face authentication for gated models (for example, IndicBERT v1).
# Set HF_TOKEN or HUGGINGFACE_HUB_TOKEN in the notebook environment after accepting
# the model terms; never paste a real token into this notebook.
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print('Hugging Face token not set; gated models require HF_TOKEN after access is granted.')
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
import sys; sys.path.insert(0, str(ROOT / 'notebooks'))
from hinglish_hate import load_hasoc2022_threads, filter_romanised

DATA = ROOT / 'data'
W    = ROOT / 'writing'
W.mkdir(parents=True, exist_ok=True)
print('data folder found:', DATA.exists(), '| root:', ROOT)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LOG  = W / 'multiseed_results.csv'
LOG.unlink(missing_ok=True)   # start clean, or you get duplicates again
print('device:', DEVICE)

# FROZEN SPLIT - loaded, never re-created
train_df = pd.read_parquet(DATA/'splits'/'bohra_train.parquet')
test_df  = pd.read_parquet(DATA/'splits'/'bohra_test.parquet')
assert len(train_df) == 3659 and len(test_df) == 915, 'split changed'
h22_r = filter_romanised(load_hasoc2022_threads(DATA), include_mixed=True)
print(f'train {len(train_df)} | test {len(test_df)} | cross {len(h22_r)}')

counts  = np.bincount(train_df['label'].values)
weights = torch.tensor(len(train_df)/(2.0*counts), dtype=torch.float).to(DEVICE)

class PostDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len=128):
        self.t=list(texts); self.y=list(labels); self.tok=tok; self.m=max_len
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(str(self.t[i]), truncation=True, padding='max_length',
                     max_length=self.m, return_tensors='pt')
        return {'input_ids':e['input_ids'].squeeze(0),
                'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.y[i], dtype=torch.long)}

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

NUM_WORKERS = 0  # Windows notebooks must not spawn Dataset workers

def run(model_name, epochs, seed, lr=2e-5, bs=16):
    set_all_seeds(seed)                      # <-- the only thing that varies
    tok = AutoTokenizer.from_pretrained(model_name)
    tr = DataLoader(PostDataset(train_df['text'], train_df['label'], tok), batch_size=bs, shuffle=True, num_workers=NUM_WORKERS)
    te = DataLoader(PostDataset(test_df['text'],  test_df['label'],  tok), batch_size=bs*2, num_workers=NUM_WORKERS)
    cr = DataLoader(PostDataset(h22_r['text'],    h22_r['label'],    tok), batch_size=bs*2, num_workers=NUM_WORKERS)

    set_all_seeds(seed)                      # again, right before the random head is made
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(tr)*epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.1*total), total)
    lossf = torch.nn.CrossEntropyLoss(weight=weights)

    t0 = time.time(); last = None
    for ep in range(1, epochs+1):
        model.train(); running = 0.0
        for b in tr:
            logits = model(input_ids=b['input_ids'].to(DEVICE),
                           attention_mask=b['attention_mask'].to(DEVICE)).logits
            loss = lossf(logits, b['labels'].to(DEVICE)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad(); running += loss.item()
        last = running/len(tr)

    def predict(dl):
        model.eval(); P,T = [],[]
        with torch.no_grad():
            for b in dl:
                lg = model(input_ids=b['input_ids'].to(DEVICE),
                           attention_mask=b['attention_mask'].to(DEVICE)).logits
                P.extend(lg.argmax(1).cpu().numpy()); T.extend(b['labels'].numpy())
        return np.array(T), np.array(P)

    yt,yp   = predict(te)
    yct,ycp = predict(cr)
    p,r,f,_ = precision_recall_fscore_support(yt, yp, zero_division=0)
    row = {'model':model_name, 'epochs':epochs, 'lr':lr, 'seed':seed,
           'macro_f1':f1_score(yt,yp,average='macro'),
           'hate_f1':f1_score(yt,yp,pos_label=1,zero_division=0),
           'f1_not':f[0], 'accuracy':accuracy_score(yt,yp),
           'cross_macro_f1':f1_score(yct,ycp,average='macro'),
           'cross_hate_f1':f1_score(yct,ycp,pos_label=1,zero_division=0),
           'final_train_loss':last, 'runtime_s':round(time.time()-t0)}

    from hinglish_hate.attack import romanisation_attack, character_attack, attack_corpus

    for atk_name, atk_fn in [('romanisation', romanisation_attack), ('character', character_attack)]:
        for rate in [0.25, 0.5, 1.0]:
            adv = attack_corpus(test_df, atk_fn, rate=rate, seed=42)
            dl  = DataLoader(PostDataset(adv['text'], adv['label'], tok), batch_size=bs*2)
            yt_a, yp_a = predict(dl)
            row[f'{atk_name}_{rate}_macro_f1'] = f1_score(yt_a, yp_a, average='macro')
            row[f'{atk_name}_{rate}_realised'] = round(adv['token_change_rate'].mean(), 3)

    prev = pd.read_csv(LOG) if LOG.exists() else pd.DataFrame()
    pd.concat([prev, pd.DataFrame([row])], ignore_index=True).to_csv(LOG, index=False)
    print(f"  seed {seed}: macro-F1 {row['macro_f1']:.3f} | hate-F1 {row['hate_f1']:.3f} "
          f"| cross {row['cross_macro_f1']:.3f} | {row['runtime_s']}s")
    del model; gc.collect(); torch.cuda.empty_cache()

    return row

CONFIGS = [('xlm-roberta-base', 3),
           ('google/muril-base-cased', 6),
           ('ai4bharat/indic-bert', 6)]     # v1, matching your epoch sweep
SEEDS   = [42, 1337, 2024]

for name, ep in CONFIGS:
    print(f'\n=== {name} | {ep} epochs ===')
    for s in SEEDS:
        try:
            run(name, ep, s)
        except Exception as e:
            print(f'  seed {s} FAILED: {type(e).__name__}: {str(e)[:150]}')

Hugging Face token not set; gated models require HF_TOKEN after access is granted.
data folder found: True | root: H:\Data\msc-hinglish-hate
device: cuda
train 3659 | test 915 | cross 4306

=== xlm-roberta-base | 3 epochs ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  seed 42: macro-F1 0.666 | hate-F1 0.589 | cross 0.456 | 90s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

[transformers] Could not extract SentencePiece model from H:\hf_cache\hub\models--ai4bharat--indic-bert\snapshots\4842dd258ecc0546f0d660b76a3b22a9c632f401\spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


  seed 1337 FAILED: ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.


[transformers] Could not extract SentencePiece model from H:\hf_cache\hub\models--ai4bharat--indic-bert\snapshots\4842dd258ecc0546f0d660b76a3b22a9c632f401\spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


  seed 2024 FAILED: ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.


In [3]:
import os
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
LOG = Path(os.environ.get('HINGLISH_HATE_MULTISeed_LOG', ROOT / 'writing' / 'multiseed_results.csv'))
if not LOG.exists():
    raise FileNotFoundError(f'Multiseed results file not found: {LOG}')

df = pd.read_csv(LOG)
summary = (df.groupby('model')
             .agg(n=('macro_f1','size'),
                  macro_mean=('macro_f1','mean'), macro_std=('macro_f1','std'),
                  hate_mean=('hate_f1','mean'),   hate_std=('hate_f1','std'),
                  cross_mean=('cross_macro_f1','mean'), cross_std=('cross_macro_f1','std'))
             .round(3).reset_index())
display(df.round(3)); display(summary)
print('\nbaseline for comparison: within 0.657 +/- 0.005 (5-fold) | cross 0.510')
df.to_csv(LOG, index=False)

,model,epochs,lr,seed,macro_f1,hate_f1,f1_not,accuracy,cross_macro_f1,cross_hate_f1,...,romanisation_0.5_macro_f1,romanisation_0.5_realised,romanisation_1.0_macro_f1,romanisation_1.0_realised,character_0.25_macro_f1,character_0.25_realised,character_0.5_macro_f1,character_0.5_realised,character_1.0_macro_f1,character_1.0_realised
0,xlm-roberta-base,3,0.0,42,0.666,0.589,0.744,0.684,0.456,0.276,...,0.636,0.381,0.631,0.762,0.644,0.18,0.651,0.356,0.654,0.71
1,xlm-roberta-base,3,0.0,1337,0.672,0.609,0.735,0.684,0.406,0.158,...,0.631,0.381,0.626,0.762,0.663,0.18,0.638,0.356,0.654,0.71
2,xlm-roberta-base,3,0.0,2024,0.657,0.573,0.741,0.678,0.451,0.270,...,0.640,0.381,0.629,0.762,0.650,0.18,0.630,0.356,0.618,0.71
3,google/muril-base-cased,6,0.0,42,0.650,0.518,0.782,0.699,0.371,0.085,...,0.643,0.381,0.639,0.762,0.650,0.18,0.632,0.356,0.589,0.71
4,google/muril-base-cased,6,0.0,1337,0.674,0.573,0.775,0.705,0.398,0.166,...,0.633,0.381,0.601,0.762,0.656,0.18,0.630,0.356,0.550,0.71
5,google/muril-base-cased,6,0.0,2024,0.678,0.588,0.769,0.704,0.440,0.227,...,0.648,0.381,0.635,0.762,0.662,0.18,0.644,0.356,0.611,0.71


,model,n,macro_mean,macro_std,hate_mean,hate_std,cross_mean,cross_std
0,google/muril-base-cased,3,0.667,0.015,0.56,0.036,0.403,0.035
1,xlm-roberta-base,3,0.665,0.008,0.59,0.018,0.438,0.027



baseline for comparison: within 0.657 +/- 0.005 (5-fold) | cross 0.510
